# OpenAI Agents SDK

Merged walkthrough of Week 2 labs:
- [1_lab1.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/1_lab1.ipynb) — Agent, Runner, traces, tools, sessions
- [2_lab2.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/2_lab2.ipynb) — multi-agent orchestration (code + LLM)
- [3_lab3.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/3_lab3.ipynb) — other models, structured outputs, guardrails
- [4_lab4.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/4_lab4.ipynb) — Deep Research (hosted web search + plan→search→report pipeline; Stage 10)

**Docs:** https://openai.github.io/openai-agents-python/

**Before running:** parent `.env` with `OPENAI_API_KEY`. Optional: `PUSHOVER_USER` / `PUSHOVER_TOKEN` for notify tools; `GOOGLE_API_KEY` for Stage 7 (Gemini free tier).

**Package name note:** install `openai-agents` (not the unrelated `agents` package on PyPI).


In [ ]:
# install openai-agents in command line
!pip3 install openai-agents

In [ ]:
# imports + env
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


## Practice arc

Work the stages in order. Each builds on the last.

| Stage | Focus | Checkpoint |
|------|--------|------------|
| 1 | SDK hello — `Agent` + `Runner` | One agent reply works |
| 2 | Observability + streaming | Trace on [OpenAI traces](https://platform.openai.com/traces) |
| 3 | Function tools (`@function_tool`) | Tool call visible in the trace |
| 4 | Memory / sessions | Name remembered across two `Runner.run` calls |
| 5 | Orchestrate by code | Parallel drafts → picker (optional send) |
| 6 | Orchestrate by LLM | Manager via agents-as-tools, then handoffs |
| 7 | Other providers / models | Non-OpenAI model (e.g. Gemini) runs via OpenAI-compatible client |
| 8 | Structured outputs | `final_output` is a Pydantic object, not free text |
| 9 | Guardrails | DIY checker and/or SDK `@output_guardrail` tripwire |
| 10 | Deep Research pipeline | Plan → parallel `WebSearchTool` → report → notify |

Delivery (email/SMTP) is optional. Prefer Pushover or a dry-run print tool so the focus stays on agents.

Stages 7–9 reuse the sales / draft-review themes from lab3 so they sit naturally after orchestration. Stage 10 maps to course [4_lab4](https://github.com/ed-donner/agents/blob/main/2_openai/4_lab4.ipynb) (Deep Research).

**Cost note:** Stage 10 uses OpenAI's paid `WebSearchTool`. See that stage's warning before you run it.


---
## Stage 1 — SDK hello

A simple `Agent` plus `Runner.run` is enough for a first loop: name, instructions, model, then a prompt.

Practice:
1. Create an agent (e.g. a joke teller)
2. `await Runner.run(agent, "...")`
3. Read `result.final_output` and `result.to_input_list()` to see the turn detail


In [ ]:
# Stage 1 — Agent + Runner.run

# setup agent
MODEL = "gpt-4o-mini"
instructions = "You are a helpful assistant that can answer questions and help with tasks."

agent = Agent(name="assistant", instructions=instructions, model=MODEL)

# run agent with question
user_input = "please help me solve this math problem: 2 + 2 = ?"
result = await Runner.run(agent, user_input)
print(result.final_output)
result.to_input_list()


---
## Stage 2 — Observability + streaming

### Trace (observability)

Wrap a run in `trace("...")` and open the OpenAI dashboard to inspect the call graph.

**Checkpoint:** find this run under https://platform.openai.com/traces

### Streaming — why it exists

`Runner.run` waits until the model finishes, then gives you the full answer in `result.final_output`. That is simple, but the notebook (or UI) looks idle until everything is done.

`Runner.run_streamed` starts the same agent loop, but **emits events as tokens arrive**. You can print (or render) text piece by piece — the same idea as ChatGPT typing live.

### How the stream cell works

1. **`Runner.run_streamed(agent, input=...)`** — kick off the run; returns a streamed result handle (do not `await` this call itself).
2. **`async for event in result.stream_events()`** — iterate every event from the run (raw model chunks, tool activity, etc.).
3. **Filter to text deltas** — keep events where `event.type == "raw_response_event"` and `event.data` is a `ResponseTextDeltaEvent`.
4. **`print(event.data.delta, end="", flush=True)`** — each `delta` is a small string fragment; concatenate them in order to rebuild the reply as it streams.

`ResponseTextDeltaEvent` comes from `openai.types.responses` — it is the SDK’s typed wrapper for “new text chunk” events on the Responses API stream.

### Contrast with Stage 1

| | `Runner.run` | `Runner.run_streamed` |
|--|--------------|------------------------|
| When you see output | After the full reply | Token by token |
| Typical access | `result.final_output` | `event.data.delta` in a loop |
| Best for | Scripts, tests, batch | Chat UIs, demos, long answers |

Same agent, same model — only the delivery of tokens changes.

### Cost note

Streaming does **not** mean more LLM calls or more agent-loop iterations. `Runner.run` and `Runner.run_streamed` do the same generation for that turn; streaming only changes how tokens are delivered over the wire.

You are billed for **input + output tokens** (and any tool rounds), not per stream event or per `delta`. Local `async for` / `print` loops are free — they just consume the stream.

Cost goes up when the answer is longer, you add tool/agent rounds, or you re-run the cell — same as without streaming.


In [ ]:
# Stage 2 — trace
user_input = "please help me solve this math problem: 2 + 2 = ?"
with trace("assitance"):

    result = await Runner.run(agent, user_input)
print(result.final_output)


In [ ]:
# Stage 2 — stream
from openai.types.responses import ResponseTextDeltaEvent

user_input = "Please tell me 5 jokes about AI Agents."

result = Runner.run_streamed(agent, input=user_input)
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


---
## Stage 3 — Function tools

Use `@function_tool` so the SDK builds the JSON schema for you. Attach the tool to an agent and ask it to notify (Pushover) or print a dry-run message.

Practice:
1. Define a small notify / push helper
2. Decorate it with `@function_tool`
3. Inspect `.params_json_schema`
4. Run an agent that must call the tool; confirm in the trace


In [ ]:
# Stage 3 — @function_tool + agent with tools
@function_tool
def notify_user(message: str) -> str:
    print('Email sent! msg:', message)
notify_user.params_json_schema

agent = Agent(name='Notification Agent', model=MODEL, tools=[notify_user])
with trace("run with function tool"):
    result = await Runner.run(agent, "Please notify the user that the email has been sent.")

    print('agent output: ', result.final_output)

---
## Stage 4 — Memory / sessions

Each `Runner.run` starts fresh unless you pass history.

Practice both approaches:
1. **Manual** — feed the next turn with `result.to_input_list()` plus the new user message
2. **Built-in** — `SQLiteSession` (in-memory is fine for the lab)

**Checkpoint:** after “Hi, my name is …”, a second run answers with that name.


In [ ]:
# Stage 4a — no memory (expect failure to recall)
agent = Agent(name='Memoryless Agent', model=MODEL)
with trace("Agent without memory - step 1"):
    result1 = await Runner.run(agent, "Hi, my name is Jordan.")
print("Result 1:", result1.final_output)

with trace("Agent without memory - step 2"):
    result2 = await Runner.run(agent, "What's my name?")
print("Result 2:", result2.final_output)

In [ ]:
# Stage 4b — manual history via to_input_list()

# After the first run, carry over the conversation manually as input history
agent = Agent(name='Manual History Agent', model=MODEL)

# Step 1: User introduces their name
with trace("Manual history - step 1"):
    result1 = await Runner.run(agent, "Hi, my name is Jordan.")
print("Result 1 (manual history):", result1.final_output)

# Step 2: Ask for the name, passing the previous conversation as history
manual_history = result1.to_input_list() + [{"role": "user", "content": "What's my name?"}]
with trace("Manual history - step 2"):
    result2 = await Runner.run(agent, manual_history)
print("Result 2 (manual history):", result2.final_output)



In [ ]:
# Stage 4c — SQLiteSession
session = SQLiteSession("Agent Memory")

# step 1: user introduces their name with passed session
with trace("Session history - step 1"):
    result1 = await Runner.run(agent, "Hi, my name is Jordan.", session=session)
print("Result 1 (session history):", result1.final_output)

# Step 2: Ask for the name, with passed session
with trace("Session history - step 2"):
    result2 = await Runner.run(agent, "What's my name?", session=session)
print("Result 2 (session history):", result2.final_output)

---
## Stage 5 — Orchestrate by code

More predictable than letting one LLM plan everything: you wire the workflow in Python.

Pattern (sales cold-email theme from lab2, or any “N drafts → pick best”):
1. Define 3 writer agents with different styles
2. Run them in parallel with `asyncio.gather`
3. Pass all drafts to a picker agent
4. Optionally attach a send/notify tool and require a tool call (`ModelSettings(tool_choice="required")`)

**Checkpoint:** one trace shows parallel writers + selection (and optional send).


In [ ]:
# Stage 5a — three writer agents + streamed sample
from agents import Agent, Runner, trace

# Define three writer agents with different styles
writer1 = Agent(
    instructions="You are a friendly and enthusiastic sales email writer. Write persuasive outreach emails in a warm, conversational tone. Make the recipient feel at ease and curious.",
    name="Friendly Writer",
    model=MODEL,
)

writer2 = Agent(
    instructions="You are a concise and professional sales email writer. Write directly to the point, using a formal and respectful business voice. Value the recipient's time.",
    name="Professional Writer",
    model=MODEL,
)

writer3 = Agent(
    instructions="You are a creative sales email writer who uses playful metaphors and humor to make the message stand out. Aim for memorability and light-heartedness, while remaining business-appropriate.",
    name="Creative Writer",
    model=MODEL,
)

# Sample email prompt
message = (
    "Draft a short cold email introducing our new AI-powered analytics tool. "
    "Audience: small business owners. Goal: Get them to book a demo."
)

In [ ]:
# Stage 5b — asyncio.gather parallel runs
import asyncio
result_drafts = None

with trace("Parallel drafts emails"):
    result_drafts = await asyncio.gather(
        Runner.run(writer1, message),
        Runner.run(writer2, message),
        Runner.run(writer3, message),
    )


In [ ]:
result_drafts


In [ ]:
# Stage 5c — picker (+ optional send tool)

# Define a picker agent to choose the best draft from the writers
picker = Agent(
    name="Draft Picker",
    instructions=(
        "You are an expert sales manager. "
        "Review the candidate emails below (each with a different style) for a cold sales outreach introducing our AI-powered analytics tool. "
        "Evaluate each draft for persuasiveness, clarity, professionalism, and likelihood to engage small business owners to book a demo. "
        "Return the best draft, with a brief explanation of your choice if useful."
    ),
    model=MODEL,
)

drafts = [draft.final_output for draft in result_drafts]
draft_str = "Draft Emails:\n\n" + "\n\n".join(drafts)

with trace("Draft Selection"):
    best = await Runner.run(picker, draft_str)
    print("Best draft:", best.final_output)

---
## Stage 6 — Orchestrate by LLM

Two collaboration styles in the OpenAI Agents SDK:

### 6a — Agents as tools
Expose writers with `agent.as_tool(...)`. A **Sales Manager** (planning agent) calls writer tools, picks the best draft, then calls send/notify.

Flow: **A → B → A** (control returns to the manager).

In [ ]:
# Stage 6— writers as tools + manager
tool_writer1 = writer1.as_tool(tool_name="writer_tool1", tool_description="Tool of writer 1")
tool_writer2 = writer2.as_tool(tool_name="writer_tool2", tool_description="Tool of writer 2")
tool_writer3 = writer3.as_tool(tool_name="writer_tool3", tool_description="Tool of writer 3")

@function_tool
def send_email(email: str) -> str:
    print('Email sent! msg:', email)
# send_email.params_json_schema

instructions = """
You are a sales manager. You have three writers and a sender. 
You will call the writers to get three drafts. 
Then you will pick the best draft and send it to the sender.
"""


In [ ]:
manager = Agent(name="LLM Sales Manager", instructions=instructions, tools=[tool_writer1, tool_writer2, tool_writer3, send_email])
with trace("Sales manager (LLM)"):
    result = await Runner.run(manager, "write me a email to send to a small business owner to book a demo")

### 6b — Handoffs
Manager drafts via tools, then **hands off** to a sender. Control passes across: **A → B**.

Handoffs can be flaky—especially on smaller models. Iterate on prompts, or prefer agents-as-tools when you need reliability.

Optional: `draw_graph(manager)` to sketch the graph.

**Checkpoint:** compare the two traces; confirm only one outbound message when send is enabled.


In [ ]:
# Stage 6b — handoffs

# leverage notify_user agent as a handoff agent
notifier_agent = Agent(name='Notification Agent', model=MODEL, tools=[notify_user])


manager = Agent(name="Handoff Sales Manager", instructions=instructions, tools=[tool_writer1, tool_writer2, tool_writer3], handoffs=[notifier_agent])
with trace("Sales manager (handoffs)"):
    result = await Runner.run(manager, "write me a email to send to a small business owner to book a demo")


---
## Stage 7 — Other providers / models

The Agents SDK is not limited to OpenAI model strings. Any **OpenAI-compatible** HTTP API can back an `Agent` in three steps:

1. Set the provider **base URL** (Gemini, Groq, OpenRouter, …)
2. Build an **`AsyncOpenAI`** client with that `base_url` + API key
3. Wrap it in **`OpenAIChatCompletionsModel`**, then pass that object as `Agent(model=...)`

For a free lab tier, **Google Gemini** is the usual choice (`GOOGLE_API_KEY` in parent `.env`). Groq / OpenRouter work the same pattern if you have keys.

Practice:
1. Create a Gemini-backed sales (or writer) agent
2. Optionally mix OpenAI + Gemini writers under one manager (as in lab3)
3. Confirm the multi-model run in a single trace

**Checkpoint:** a non-OpenAI model produces a draft (and shows up in https://platform.openai.com/traces when tracing is enabled).

Docs / guide: course Guide 9 for compatible base URLs; SDK model docs.


In [ ]:
# Stage 7 — Gemini (or other OpenAI-compatible) model
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel
import os

load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY is not set")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"


gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=GOOGLE_API_KEY)
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",  # pick a current free-tier model id
    openai_client=gemini_client,
)
gemini_agent = Agent(name="Gemini Writer", instructions="you are a sales email writer", model=gemini_model)
with trace("Other provider"):
    result = await Runner.run(gemini_agent, "Write a short sales email")
print(result.final_output)


In [ ]:
# Stage 7b (optional) — mix providers under one manager
tool_gemini = gemini_agent.as_tool(tool_name="gemini_writer", tool_description="Gemini model writer")
tool_openai = writer1.as_tool(tool_name="openai_writer", tool_description="OpenAI model writer")
manager = Agent(name="Sales Manager", instructions="Draft emails using both Gemini and OpenAI models, then decide which is best and notify the user", tools=[tool_gemini, tool_openai, notify_user], model=MODEL)
with trace("Sales Manager across different models"):
    result = await Runner.run(manager, "write me a sales email to send to a small business owner to book a demo")
    print(result.final_output)

---
## Stage 8 — Structured outputs

LLMs default to natural-language text. **Structured outputs** make the model return data that maps to a **Python object** (usually a Pydantic `BaseModel`):

1. Define the schema as a `BaseModel` subclass
2. Pass `output_type=YourModel` on the `Agent`
3. Read fields on `result.final_output` (e.g. `review.is_professional`) — no brittle string parsing

Under the hood: schema + JSON (and, on supporting providers, **constrained decoding**). Not every provider supports structured outputs equally well.

Use this when downstream code must branch on fields — reviewers, routers, policy judges, email QA.

**Checkpoint:** run a checker agent on a sample email and print typed fields from `final_output`.


In [ ]:
# Stage 8 — Pydantic schema + output_type
from pydantic import BaseModel, Field

class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="Sentence count in the body (excl. greeting/signature)")
    contains_placeholders: bool = Field(description="Whether the email still has placeholders")

EmailReview.model_json_schema()


In [ ]:
# Stage 8b — checker agent returns EmailReview
checker = Agent(
    name="Checker",
    instructions="You review potential sales emails",
    model=MODEL,
    output_type=EmailReview,
)

# draft a sales email
sample_email = """
Dear [Customer Name],

I'm reaching out to discuss our new AI-powered analytics tool. 
We're excited to introduce a solution that can help you optimize your operations and drive growth.

Best regards,

[Your Name]
"""

result = await Runner.run(checker, sample_email)
review = result.final_output
print('is_professional:', review.is_professional, '\ncontains_placeholders:', review.contains_placeholders, '\nnumber_of_sentences:', review.number_of_sentences)


---
## Stage 9 — Guardrails

**Guardrails** are runtime **controls** that stop undesirable behavior (bad input, bad output, or unsafe tool use) — often with logic or a second LLM call.

They are one piece of a broader **agent harness** (loop, tools, memory, eval, gates…). Your `2_guardrails` live policy gate (`classify` / `judge`) is the same *idea* as a guardrail; the offline probe suite is a small eval helper beside that gate.

### Two ways to practice

**A — DIY (portable):** after generating an email, `Runner.run(checker, email)` and branch in Python. Works in any framework; closest to `2_guardrails` policy.

**B — SDK guardrails:** `@output_guardrail` (also input / tool). Docs: https://openai.github.io/openai-agents-python/guardrails/

**SDK gotcha:** input guardrails run only for the **first input to the first agent** in `Runner.run()`; output guardrails only for the **final output of the last agent**. Guardrails on middle agents may never run.

**Checkpoint:** block or flag an unprofessional / placeholder-filled email — once with DIY, optionally once with `@output_guardrail`.


In [ ]:
# Stage 9a — DIY guardrail (explicit second run)
writer = Agent(name="Writer", instructions="You are a sales email writer", model=MODEL)
draft = (await Runner.run(writer, "Write a cold sales email")).final_output
review = (await Runner.run(checker, draft)).final_output  # checker from Stage 8
if review.contains_placeholders or not review.is_professional:
    print("Blocked:", review, '\n\n**Draft:**\n\n', draft)
else:
    print("OK to send, draft:", draft)


In [ ]:
# Stage 9b — SDK @output_guardrail (optional)
from agents import output_guardrail, GuardrailFunctionOutput, OutputGuardrailTripwireTriggered

@output_guardrail
async def email_guardrail(ctx, agent, message):
    """SDK output-guardrail callback. All three args are required by the decorator.
    Args:
        ctx: Run context for this Runner.run (includes ctx.context you can pass through).
        agent: The Agent instance that produced this output (e.g. agent.name); often unused.
        message: The agent output being checked (usually text / final message content).
    """
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=is_problem,
    )

gated_writer = Agent(
    name="Gated Writer",
    instructions="write a cold sales email",
    model=MODEL,
    output_guardrails=[email_guardrail],
)
with trace("Guardrail"):
    try:
        result = await Runner.run(gated_writer, "Write a cold sales email")
    except OutputGuardrailTripwireTriggered as e:
        print("Blocked draft:", e.guardrail_result.agent_output)
        print("Why (output_info):", e.guardrail_result.output.output_info)

---
## Stage 10 — Deep Research (lab 4)

Capstone pattern from [4_lab4.ipynb](https://github.com/ed-donner/agents/blob/main/2_openai/4_lab4.ipynb): a **code-orchestrated** research pipeline (not an LLM manager).

**Four agents:**
1. **Planner** — structured list of search queries (`WebSearchPlan`)
2. **Search** — hosted `WebSearchTool` + `ModelSettings(tool_choice="required")`
3. **Writer** — long markdown report (`ReportData`)
4. **Emailer** — notify/send tool with the report

**What's new vs Stages 1–9:** OpenAI **hosted tools** (`WebSearchTool`), forcing tool use with `tool_choice="required"`, nested Pydantic plans with `Field(description=...)`, and a plan → fan-out → synthesize → deliver pipeline under one `trace`.

### WARNING — `WebSearchTool` costs money

`WebSearchTool` is a **paid** OpenAI hosted tool (on the order of **~$0.01 per call**; OpenAI may bill **multiple searches** inside a single call, so it can cost more). A full Stage 10 run with `HOW_MANY_SEARCHES = 3` can easily be several cents of tool fees **on top of** normal model tokens.

- Pricing (Tools section): https://developers.openai.com/api/docs/pricing
- Hosted tools docs: https://openai.github.io/openai-agents-python/tools/#hosted-tools

**Skip the run cells below if you do not want to spend money.** Reading the code still teaches the pattern.

**Checkpoint:** one traced run plans searches, gathers summaries in parallel, writes a report, and notifies (dry-run print is fine).


In [ ]:
# Stage 10 — imports + knobs (PAID: WebSearchTool)
# Only run the cells below if you accept OpenAI Web Search tool charges.
# Pricing: https://developers.openai.com/api/docs/pricing

from agents import WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
import asyncio

HOW_MANY_SEARCHES = 3  # keep small — each search agent run can bill WebSearchTool
RESEARCH_QUERY = "Most popular AI Agent frameworks in 2026"


In [ ]:
# Stage 10a — Search Agent (hosted WebSearchTool + tool_choice required)
SEARCH_INSTRUCTIONS = """
You are a research assistant. Given a search term, search the web for that term and
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 words.
Capture the main points. Reply only with the summary.
""".strip()

search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool()],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)

# Optional smoke test (1 paid WebSearchTool call) — comment out if skipping spend
# result = await Runner.run(search_agent, RESEARCH_QUERY)
# display(Markdown(result.final_output))


In [ ]:
# Stage 10b — Planner (nested structured output)
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query."
    )


planner_agent = Agent(
    name="Planner Agent",
    instructions=(
        f"You are a research assistant. Given a user query, come up with a set of web searches "
        f"to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."
    ),
    model=MODEL,
    output_type=WebSearchPlan,
)


In [ ]:
# Stage 10c — Writer (ReportData) + Emailer (reuse notify_user from Stage 3)
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="Writer Agent",
    instructions="""
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed.
Aim for a solid multi-section report (several hundred words is fine for this lab).
""".strip(),
    model=MODEL,
    output_type=ReportData,
)

email_agent = Agent(
    name="Email Agent",
    instructions=(
        "You are provided with a detailed report. Use your tool to notify the user, "
        "with a short subject-style first line and the report summary in the message."
    ),
    tools=[notify_user],  # from Stage 3 — dry-run print; swap for Pushover/SMTP if you prefer
    model=MODEL,
)


In [ ]:
# Stage 10d — code orchestration helpers (plan → parallel search → write → notify)
async def search(item: WebSearchItem) -> str:
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output


async def run_searches(query: str) -> list[str]:
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches (each may bill WebSearchTool)")
    results = await asyncio.gather(*[search(item) for item in searches])
    print("Finished searching")
    return list(results)


async def write_report(query: str, search_results: list[str]) -> ReportData:
    print("Writing report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output


async def send_report(report: ReportData) -> str:
    print("Notifying...")
    result = await Runner.run(
        email_agent,
        f"Short summary:\n{report.short_summary}\n\nReport:\n{report.markdown_report}",
    )
    print("Notify step done")
    return result.final_output


In [ ]:
# Stage 10e — showtime (PAID: runs HOW_MANY_SEARCHES WebSearchTool calls)
# Skip this cell if you do not want to spend money.
# Pricing: https://developers.openai.com/api/docs/pricing

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(RESEARCH_QUERY)
    report = await write_report(RESEARCH_QUERY, search_results)
    await send_report(report)
    print("Hooray!")
    display(Markdown(report.short_summary))
    display(Markdown(report.markdown_report))
    print("Follow-ups:", report.follow_up_questions)

# Inspect the full workflow: https://platform.openai.com/traces


---
## Stretch (optional)

After stages 1–10 feel solid:

- Rebuild a Week 1 idea (digital twin or checklist loop) on this SDK
- Swap Pushover/dry-run for SendGrid or Resend; handle a reply and keep an SDR conversation going
- Add another refine pass (draft → critique → rewrite) with code or LLM orchestration
- Package Stage 10 Deep Research into `agent/` modules + a Gradio entrypoint (mirror Sales Email Studio)
- **Sandbox agents** (lab3 optional) — local workspace to edit files / run commands
- **MCP teaser** (lab3 optional) — `MCPServerStreamableHttp` for remote tools

Sales Email Studio already lives in `app.py` + `agent/`; Stage 10 stays notebook-only until you extract it.
